In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

True
Tesla T4


In [2]:
import os
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
print("WANDB_API_KEY is set:", bool(os.environ.get("WANDB_API_KEY")))

WANDB_API_KEY is set: True


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [12]:
%cd /content

!rm -rf /content/sixray-kd
!git clone https://github.com/AnnyNny/sixray-kd.git /content/sixray-kd

%cd /content/sixray-kd
!git checkout anna-local-work

!git status
!git log --oneline -5

/content
Cloning into '/content/sixray-kd'...
remote: Enumerating objects: 171, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 171 (delta 61), reused 148 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (171/171), 7.85 MiB | 25.37 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/sixray-kd
Branch 'anna-local-work' set up to track remote branch 'anna-local-work' from 'origin'.
Switched to a new branch 'anna-local-work'
On branch anna-local-work
Your branch is up to date with 'origin/anna-local-work'.

nothing to commit, working tree clean
d3b58d5 (HEAD -> anna-local-work, origin/anna-local-work) fix
a49b1a1 fix init
f7c296d fix init
85763e6 Merge branch 'anna-local-work' of https://github.com/AnnyNny/sixray-kd into anna-local-work
9e4ed4f add 40grid ablation


In [5]:

!bash /content/sixray-kd/scripts/setup_colab.sh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
Dataset 'subset_clean.zip' extracted to /content/data


In [6]:
!pip install -q torchmetrics pycocotools wandb timm evaluate transformers

In [7]:
from pathlib import Path

paths = [
    Path("/content/data/train/images"),
    Path("/content/data/train.json"),
    Path("/content/data/test/images"),
    Path("/content/data/test.json"),
    Path("/content/drive/MyDrive/DatasetAPAI/SIXray_Project/splits/split_seed42_train10500_val300pos1200neg.json"),
]

for path in paths:
    print(path, "OK" if path.exists() else "MISSING")

/content/data/train/images OK
/content/data/train.json OK
/content/data/test/images OK
/content/data/test.json OK
/content/drive/MyDrive/DatasetAPAI/SIXray_Project/splits/split_seed42_train10500_val300pos1200neg.json OK


In [8]:
"""%env SIXRAY_ABLATION=grid40_layer3
!python students/anna_student_resnet18/scripts/smoke_test_student.py"""

'%env SIXRAY_ABLATION=grid40_layer3\n!python students/anna_student_resnet18/scripts/smoke_test_student.py'

In [13]:
%env SIXRAY_ABLATION=grid40_layer3
!python students/anna_student_resnet18/scripts/train_student.py

env: SIXRAY_ABLATION=grid40_layer3
Device: cuda
Use AMP: True
Ablation: grid40_layer3
Ablation description: Ablation with higher resolution 40x40 detection grid using resnet layer3

Building datasets...
Loaded split: /content/drive/MyDrive/DatasetAPAI/SIXray_Project/splits/split_seed42_train10500_val300pos1200neg.json
Train: 10500
Val: 1500
Test: 13246
Dataset: /content/data/train.json
Images folder: /content/data/train/images
Images used here: 10500
Dataset: /content/data/train.json
Images folder: /content/data/train/images
Images used here: 1500
Dataset: /content/data/test.json
Images folder: /content/data/test/images
Images used here: 13246

Building dataloaders...

Building model...
Total parameters: 12362836
Trainable parameters: 12362836

Initializing W&B...
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: anna_smetanina (anna_smetanina-universi

In [14]:
%env SIXRAY_ABLATION=grid40_layer3
!python students/anna_student_resnet18/scripts/evaluate_student.py

env: SIXRAY_ABLATION=grid40_layer3
Device: cuda
Use AMP: True

Building datasets...
Loaded split: /content/drive/MyDrive/DatasetAPAI/SIXray_Project/splits/split_seed42_train10500_val300pos1200neg.json
Train: 10500
Val: 1500
Test: 13246
Dataset: /content/data/train.json
Images folder: /content/data/train/images
Images used here: 10500
Dataset: /content/data/train.json
Images folder: /content/data/train/images
Images used here: 1500
Dataset: /content/data/test.json
Images folder: /content/data/test/images
Images used here: 13246

Building dataloaders...

Building model...
Total parameters: 12362836
Trainable parameters: 12362836

Loading checkpoint:
/content/drive/MyDrive/DatasetAPAI/SIXray_Project/student_checkpoints_anna/ablations/grid40_layer3/student_resnet18_yolo2_local_offsets_grid40_layer3_best_map_50.pth
Loaded epoch: 19

Evaluating val

val loss:
{'total_loss': 0.2975823944689185, 'objectness_loss': 0.01890290337597283, 'bbox_loss': 0.0048250805602098506, 'class_loss': 0.2304286

In [15]:
import json
from pathlib import Path

path = Path("/content/drive/MyDrive/DatasetAPAI/SIXray_Project/student_checkpoints_anna/ablations/grid40_layer3/student_resnet18_eval_metrics.json")

with open(path, "r") as f:
    results = json.load(f)

print("VAL")
print("mAP:", results["val"]["detection"]["map"])
print("mAP@50:", results["val"]["detection"]["map_50"])
print("mAP@75:", results["val"]["detection"]["map_75"])
print("per-class:", results["val"]["detection"]["map_per_class"])

print("\nTEST")
print("mAP:", results["test"]["detection"]["map"])
print("mAP@50:", results["test"]["detection"]["map_50"])
print("mAP@75:", results["test"]["detection"]["map_75"])
print("per-class:", results["test"]["detection"]["map_per_class"])

VAL
mAP: 0.2611795663833618
mAP@50: 0.5376765131950378
mAP@75: 0.22287246584892273
per-class: [0.4990885257720947, 0.2662309408187866, 0.11716476082801819, 0.24726642668247223, 0.1761472374200821]

TEST
mAP: 0.23174285888671875
mAP@50: 0.4640536606311798
mAP@75: 0.20549289882183075
per-class: [0.5206465125083923, 0.19479306042194366, 0.10255725681781769, 0.1926833540201187, 0.14803405106067657]


In [16]:
import json
import pandas as pd
from pathlib import Path

class_names = ["gun", "knife", "wrench", "pliers", "scissors"]

baseline_path = Path("/content/drive/MyDrive/DatasetAPAI/SIXray_Project/student_checkpoints_anna/student_resnet18_eval_metrics.json")
one_box_path = Path("/content/drive/MyDrive/DatasetAPAI/SIXray_Project/student_checkpoints_anna/ablations/one_box/student_resnet18_eval_metrics.json")
grid40_layer3_path = Path("/content/drive/MyDrive/DatasetAPAI/SIXray_Project/student_checkpoints_anna/ablations/grid40_layer3/student_resnet18_eval_metrics.json")
with open(baseline_path, "r") as f:
    baseline = json.load(f)

with open(one_box_path, "r") as f:
    one_box = json.load(f)

with open(grid40_layer3_path, "r") as f:
    grid40_layer3 = json.load(f)


def get_detection_metrics(result, split="test"):
    split_data = result[split]
    if "detection" in split_data:
        return split_data["detection"]
    if all(k in split_data for k in ["map", "map_50", "map_75"]):
        return split_data


rows = []

for name, k, result in [
    ("Baseline K=2", 2, baseline),
    ("One-box K=1", 1, one_box),
    ("Grid40 layer3", 3, grid40_layer3),
]:
    test_det = get_detection_metrics(result, split="test")

    row = {
        "model": name,
        "boxes_per_cell": k,
        "mAP": test_det["map"],
        "mAP@50": test_det["map_50"],
        "mAP@75": test_det["map_75"],
    }

    for class_name, value in zip(class_names, test_det["map_per_class"]):
        row[f"AP_{class_name}"] = value

    rows.append(row)

df = pd.DataFrame(rows)
df

,model,boxes_per_cell,mAP,mAP@50,mAP@75,AP_gun,AP_knife,AP_wrench,AP_pliers,AP_scissors
0,Baseline K=2,2,0.292397,0.553677,0.274398,0.600464,0.217004,0.175816,0.231006,0.237693
1,One-box K=1,1,0.310381,0.549394,0.309042,0.623167,0.265045,0.172125,0.262255,0.229316
2,Grid40 layer3,3,0.231743,0.464054,0.205493,0.520647,0.194793,0.102557,0.192683,0.148034


In [17]:
df.to_csv("/content/drive/MyDrive/DatasetAPAI/SIXray_Project/student_checkpoints_anna/grid40_ablation_comparison.csv", index=False)